# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
spaceship.isna().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
spaceship = spaceship.dropna()

spaceship["Cabin"] = spaceship["Cabin"].str[0]

spaceship = spaceship.drop(columns=["PassengerId", "Name"])

spaceship = pd.get_dummies(spaceship, columns=["HomePlanet", "Cabin", "Destination"], drop_first=True)

spaceship = spaceship.astype(int)

spaceship.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0,39,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1
1,0,24,0,109,9,25,549,44,1,0,0,0,0,0,0,1,0,0,0,1
2,0,58,1,43,3576,0,6715,49,0,1,0,0,0,0,0,0,0,0,0,1
3,0,33,0,0,1283,371,3329,193,0,1,0,0,0,0,0,0,0,0,0,1
4,0,16,0,303,70,151,565,2,1,0,0,0,0,0,0,1,0,0,0,1


In [6]:
X = spaceship.drop(columns=["Transported"])
y = spaceship["Transported"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (5284, 19)
Test: (1322, 19)


In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

X_train_scaled.head()

,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e
0,0.0,0.316456,0.0,0.0,0.056116,0.000000,0.028650,0.030094,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.455696,0.0,0.0,0.088015,0.135232,0.124911,0.000049,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.0,0.430380,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,0.468354,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.0,0.278481,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_temp = RandomForestClassifier(random_state=42)
rf_temp.fit(X_train_scaled, y_train)

importances = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "importance": rf_temp.feature_importances_
}).sort_values(by="importance", ascending=False)

importances

,feature,importance
1,Age,0.162207
6,Spa,0.134212
7,VRDeck,0.124942
3,RoomService,0.116671
0,CryoSleep,0.110718
4,FoodCourt,0.108232
5,ShoppingMall,0.098810
8,HomePlanet_Europa,0.028731
15,Cabin_G,0.020590
18,Destination_TRAPPIST-1e,0.017466


In [9]:
chosen_columns = importances[importances["importance"] > 0.01]["feature"].tolist()
print("Chosen columns:", chosen_columns)

X_train_sel = X_train_scaled[chosen_columns]
X_test_sel = X_test_scaled[chosen_columns]

Chosen columns: ['Age', 'Spa', 'VRDeck', 'RoomService', 'CryoSleep', 'FoodCourt', 'ShoppingMall', 'HomePlanet_Europa', 'Cabin_G', 'Destination_TRAPPIST-1e', 'Cabin_F', 'HomePlanet_Mars', 'Cabin_E']


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [10]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_sel, y_train)

y_pred = rf.predict(X_test_sel)

- Evaluate your model

In [13]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_base = accuracy_score(y_test, y_pred)
print("Accuracy base model:", round(accuracy_base, 4))

print(classification_report(y_test, y_pred))

Accuracy base model: 0.8094
              precision    recall  f1-score   support

           0       0.80      0.81      0.81       653
           1       0.82      0.80      0.81       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [14]:
param_grid = {
    "n_estimators": [100, 200],       # número de árvores
    "max_depth": [None, 10, 20],      # profundidade máxima de cada árvore
    "min_samples_split": [2, 5, 10],  # mínimo de linhas para dividir um nó
    "min_samples_leaf": [1, 2, 4]     # mínimo de linhas em cada folha
}

- Run Grid Search

In [17]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sel, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best accuracy in cross-validation:", round(grid_search.best_score_, 4))

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best accuracy in cross-validation: 0.7984


- Evaluate your model

In [18]:
best_rf = grid_search.best_estimator_

y_pred_best = best_rf.predict(X_test_sel)

accuracy_best = accuracy_score(y_test, y_pred_best)
print(classification_report(y_test, y_pred_best))

print("Accuracy base model:   ", round(accuracy_base, 4))
print("Accuracy after Grid:", round(accuracy_best, 4))
print("Improvement:               ", round(accuracy_best - accuracy_base, 4))

              precision    recall  f1-score   support

           0       0.82      0.81      0.81       653
           1       0.81      0.82      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322

Accuracy base model:    0.8094
Accuracy after Grid: 0.8147
Improvement:                0.0053
